# SkylineGeolocation — Full Parameter Study

Off-grid evaluation using Street View (primary) + Synthetic (upper-bound reference).

**Phases:**
1. Setup: clone repo, mount Drive, symlink data
2. Crop panos → perspective images
3. Segment crops → sky masks  
4. **🛑 GATE 1**: Review crops
5. **🛑 GATE 2**: Review masks
6. Baseline evaluation (SV + Synthetic)
7. Parameter sweep
8. DB parameter study (90m DEM)
9. Generate plots

All checkpoints saved to `MyDrive/eval_study/`. Re-run skips completed work.

In [ ]:
import os, shutil
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
BRANCH = 'main'

if (REPO / 'src').exists():
    print('Repo already at', REPO)
else:
    print('Cloning repo...')
    if REPO.is_file() or REPO.is_symlink():
        REPO.unlink()
    elif REPO.is_dir():
        shutil.rmtree(REPO)
    REPO.parent.mkdir(parents=True, exist_ok=True)
    import subprocess
    subprocess.run(
        ['git', 'clone', '-b', BRANCH, 'https://github.com/pxrxp/SkylineGeolocation.git', str(REPO)],
        check=True, capture_output=True, text=True
    )
    print('Cloned. Branch:', BRANCH)

os.chdir(REPO)
import sys
sys.path.insert(0, str(REPO))
print('Ready at', REPO)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
STUDY_DIR = DRIVE / 'eval_study'
STUDY_DIR.mkdir(parents=True, exist_ok=True)
print('Drive:', STUDY_DIR)

In [ ]:
links = [
    (DRIVE / 'skyline_db.parquet',
     REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'),
    (DRIVE / 'synthetic_dataset/ground_truth.json',
     REPO / 'data/synthetic_dataset/ground_truth.json'),
    (DRIVE / 'synthetic_dataset/images',
     REPO / 'data/synthetic_dataset/images'),
    (DRIVE / 'synthetic_dataset/masks',
     REPO / 'data/synthetic_dataset/masks'),
    (DRIVE / 'synthetic_dataset/predicted_masks',
     REPO / 'data/synthetic_dataset/predicted_masks'),
    (DRIVE / 'sky_segmentation_unet_model.pth',
     REPO / 'data/sky_segmentation_unet_model.pth'),
    (DRIVE / 'street_view',
     REPO / 'data/street_view'),
]

for src, dst in links:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(src, dst)
        print(f'Linked {src.name}')
    elif dst.exists():
        print(f'{dst.name}: already linked')
    else:
        print(f'MISSING: {src}')

In [ ]:
db_path = REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'
if db_path.exists():
    size = db_path.stat().st_size
    with open(db_path, 'rb') as f:
        f.seek(0); hdr = f.read(4)
        f.seek(-8, 2); ftr = f.read()
    if hdr != b'PAR1' or ftr[-4:] != b'PAR1':
        raise SystemExit('BAD: DB corrupted')
    print(f'DB OK: {size / 1e6:.0f} MB')
else:
    raise SystemExit(f'MISSING: {db_path}')

## Phase 1: Crop Panos
Overwrite `data/street_view/images/` with corrected perspective crops.

In [ ]:
import json
import csv
import numpy as np
from pathlib import Path
from PIL import Image
import sys, os, time

sys.path.insert(0, str(REPO))
from src.streetview_utils import slice_perspective

PANOS_DIR = REPO / 'data/street_view/panos'
IMAGES_DIR = REPO / 'data/street_view/images'
GT_PATH = REPO / 'data/street_view/ground_truth.json'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

with open(GT_PATH) as f:
    gt = json.load(f)

# Load GSV metadata for pitch/roll (GT uses cam_R_tilt → derive pitch/roll)
sv_meta = {}
with open(REPO / 'data/street_view/panos_metadata.csv') as f:
    for row in csv.DictReader(f):
        try:
            sv_meta[row['id']] = {
                'heading': float(row['heading']) if row['heading'] else 0.0,
                'pitch': float(row['pitch']) if row['pitch'] else 0.0,
                'roll': float(row['roll']) if row['roll'] else 0.0,
            }
        except (ValueError, KeyError):
            pass

# Checkpoint: track which panos are already cropped
ckpt_path = STUDY_DIR / 'crops_done.json'
if ckpt_path.exists():
    done = set(json.loads(ckpt_path.read_text()))
else:
    done = set()

need = [sid for sid in gt if sid not in done]
print(f'Total: {len(gt)}, Done: {len(done)}, Need: {len(need)}')

t0 = time.time()
for i, sid in enumerate(need):
    v = gt[sid]
    pano_path = PANOS_DIR / f'{sid}.jpg'
    if not pano_path.exists():
        continue

    # Extract pitch/roll from cam_R_tilt
    R = np.array(v['cam_R_tilt'])
    pitch = float(np.degrees(np.arcsin(np.clip(-R[2, 1], -1, 1))))
    roll = float(np.degrees(np.arctan2(R[2, 0], R[2, 2])))

    crop = slice_perspective(
        str(pano_path),
        heading_deg=v['true_heading_deg'],
        pitch_deg=pitch,
        roll_deg=roll,
        fov_y_deg=v['fov_y_deg'],
        out_w=1080, out_h=720,
    )
    crop.save(IMAGES_DIR / f'{sid}.png')
    done.add(sid)

    if (i + 1) % 100 == 0:
        ckpt_path.write_text(json.dumps(list(done)))
        elapsed = time.time() - t0
        print(f'  {i+1}/{len(need)}: {elapsed:.0f}s elapsed', flush=True)

ckpt_path.write_text(json.dumps(list(done)))
print(f'Done: {len(done)}/{len(gt)} crops, {time.time()-t0:.0f}s total')

In [ ]:
# Verify crop quality: show random samples for user review
import matplotlib.pyplot as plt
import random
random.seed(42)
sample_ids = random.sample(list(gt.keys()), min(12, len(gt)))

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle('STOP — Review these crops before continuing', fontsize=16, color='red')
for ax, sid in zip(axes.flat, sample_ids):
    img_path = IMAGES_DIR / f'{sid}.png'
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    ax.set_title(sid[:20], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Are these crops acceptable? (horizon visible, sky on top, terrain on bottom)')

# 🛑 GATE 1: STOP — Wait for user confirmation that crops are acceptable.
**Do NOT proceed past this point until the user reviews the crop samples above.**

If crops look wrong (sky on bottom, flipped, wrong heading), we fix `slice_perspective` first.

## Phase 2: Segment Crops → Sky Masks
Run U-Net on all crops. Save masks to `data/street_view/masks/`.

In [ ]:
import torch
from src.segmentation import load_segmentation_model, segment_image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
model = load_segmentation_model(str(REPO / 'data/sky_segmentation_unet_model.pth'), device)

MASKS_DIR = REPO / 'data/street_view/masks'
MASKS_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint
seg_ckpt_path = STUDY_DIR / 'segmentation_done.json'
if seg_ckpt_path.exists():
    seg_done = set(json.loads(seg_ckpt_path.read_text()))
else:
    seg_done = set()

all_images = sorted(IMAGES_DIR.glob('*.png'))
need_seg = [img for img in all_images if img.stem not in seg_done]
print(f'Total images: {len(all_images)}, Done: {len(seg_done)}, Need: {len(need_seg)}')

t0 = time.time()
for i, img_path in enumerate(need_seg):
    mask_path = MASKS_DIR / img_path.name
    if not mask_path.exists():
        segment_image(model, str(img_path), str(mask_path), device)
    seg_done.add(img_path.stem)

    if (i + 1) % 100 == 0:
        seg_ckpt_path.write_text(json.dumps(list(seg_done)))
        elapsed = time.time() - t0
        print(f'  {i+1}/{len(need_seg)}: {elapsed:.0f}s elapsed', flush=True)

seg_ckpt_path.write_text(json.dumps(list(seg_done)))
print(f'Done: {len(seg_done)} masks, {time.time()-t0:.0f}s total')

In [ ]:
# Show crop + mask pairs for user review
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle('STOP — Review these crop+mask pairs before continuing', fontsize=16, color='red')
for ax, sid in zip(axes.flat, sample_ids):
    img_path = IMAGES_DIR / f'{sid}.png'
    mask_path = MASKS_DIR / f'{sid}.png'
    if img_path.exists() and mask_path.exists():
        img = np.array(Image.open(img_path))
        mask = np.array(Image.open(mask_path).convert('L'))
        overlay = img.copy()
        overlay[mask > 127] = overlay[mask > 127] * 0.3 + np.array([100, 180, 255]) * 0.7
        ax.imshow(overlay.astype(np.uint8))
    ax.set_title(sid[:20], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Are these masks acceptable? (sky=blue overlay, terrain=original)')

# 🛑 GATE 2: STOP — Wait for user confirmation that masks are acceptable.
**Do NOT proceed past this point until the user reviews the mask samples above.**

## Phase 3: Baseline Evaluation

### 3A: Street View Baseline (primary, off-grid)

In [ ]:
import json
import pandas as pd
from src.evaluation import run_evaluation

sv_ckpt_dir = STUDY_DIR / 'sv_baseline'
sv_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Check if already done
sv_result_path = STUDY_DIR / 'results_sv_baseline.csv'
if sv_result_path.exists():
    df_sv = pd.read_csv(sv_result_path)
    print(f'Already done: {len(df_sv)} rows loaded from {sv_result_path}')
else:
    print('Running SV baseline (all samples, default params)...')
    df_sv, summary_sv = run_evaluation(
        ground_truth_path=str(REPO / 'data/street_view/ground_truth.json'),
        db_path=str(db_path),
        masks_dir=str(REPO / 'data/street_view/masks'),
        use_altimeter=True,
        use_compass=True,
        limit=0,
        sample_batch_size=8,
        top_k=30,
        dtw_window=15,
        correct_dist_m=500.0,
        chunk_rows=4000,
        spatial_stride=5,
        checkpoint_dir=str(sv_ckpt_dir),
    )
    df_sv.to_csv(sv_result_path, index=False)
    print(f'\nSV Baseline Summary:')
    for k, v in summary_sv.items():
        print(f'  {k}: {v}')

### 3B: Synthetic Baseline (upper-bound reference, on-grid)

In [ ]:
syn_result_path = STUDY_DIR / 'results_syn_baseline.csv'
if syn_result_path.exists():
    df_syn = pd.read_csv(syn_result_path)
    print(f'Already done: {len(df_syn)} rows')
else:
    print('Running synthetic baseline...')
    df_syn, summary_syn = run_evaluation(
        ground_truth_path=str(REPO / 'data/synthetic_dataset/ground_truth.json'),
        db_path=str(db_path),
        masks_dir=str(REPO / 'data/synthetic_dataset/predicted_masks'),
        use_altimeter=True,
        use_compass=True,
        limit=0,
        sample_batch_size=8,
        top_k=30,
        dtw_window=15,
        correct_dist_m=500.0,
        chunk_rows=4000,
        spatial_stride=5,
    )
    df_syn.to_csv(syn_result_path, index=False)
    print(f'\nSynthetic Baseline Summary (upper-bound, on-grid):')
    for k, v in summary_syn.items():
        print(f'  {k}: {v}')

## Phase 4: Parameter Sweep
Single-DB-pass sweep on a subset for speed.

In [ ]:
SWEEP_SUBSET = 500  # use first N SV samples with masks for sweep

sweep_configs = {
    # Isolation: altimeter
    'alt_only':         {'use_altimeter': True,  'use_compass': False},
    'compass_only':     {'use_altimeter': False, 'use_compass': True},
    'no_sensors':       {'use_altimeter': False, 'use_compass': False},
    'both_default':     {'use_altimeter': True,  'use_compass': True},

    # Compass tolerance
    'compass_tol_5':    {'compass_tolerance_deg': 5.0},
    'compass_tol_10':   {'compass_tolerance_deg': 10.0},
    'compass_tol_40':   {'compass_tolerance_deg': 40.0},

    # Height tolerance
    'height_tol_50':    {'height_tolerance_m': 50.0},
    'height_tol_100':   {'height_tolerance_m': 100.0},
    'height_tol_500':   {'height_tolerance_m': 500.0},

    # DTW window
    'dtw_5':            {'dtw_window': 5},
    'dtw_10':           {'dtw_window': 10},
    'dtw_20':           {'dtw_window': 20},

    # Spatial stride
    'stride_3':         {'spatial_stride': 3},
    'stride_10':        {'spatial_stride': 10},
    'stride_20':        {'spatial_stride': 20},

    # Weights
    'w_value_heavy':    {'weights': (0.5, 0.25, 0.25)},
    'w_grad_heavy':     {'weights': (0.25, 0.5, 0.25)},
    'w_d2_heavy':       {'weights': (0.25, 0.25, 0.5)},

    # Combination: tight
    'tight_sensors':    {'use_altimeter': True, 'use_compass': True,
                         'compass_tolerance_deg': 5.0, 'height_tolerance_m': 50.0},

    # Combination: loose
    'loose_sensors':    {'use_altimeter': True, 'use_compass': True,
                         'compass_tolerance_deg': 40.0, 'height_tolerance_m': 500.0},
}
print(f'{len(sweep_configs)} configs defined')

In [ ]:
import numpy as np
from src.evaluation import run_parameter_sweep

sweep_ckpt = STUDY_DIR / 'sweep_results.json'
if sweep_ckpt.exists():
    print('Sweep already done — loading results')
    import json
    with open(sweep_ckpt) as f:
        all_summaries = json.load(f)
else:
    # Create a temporary GT file for the subset
    sv_gt = json.load(open(REPO / 'data/street_view/ground_truth.json'))
    subset_ids = list(sv_gt.keys())[:SWEEP_SUBSET]
    subset_gt = {k: sv_gt[k] for k in subset_ids}
    subset_gt_path = STUDY_DIR / 'gt_subset.json'
    subset_gt_path.write_text(json.dumps(subset_gt))

    # Masks dir for subset — filter to only IDs with masks
    mask_files = {p.stem for p in (REPO / 'data/street_view/masks').glob('*.png')}
    subset_ids = [s for s in subset_ids if s in mask_files]
    subset_gt = {k: sv_gt[k] for k in subset_ids}
    subset_gt_path.write_text(json.dumps(subset_gt))
    print(f'Subset: {len(subset_ids)} samples with masks')

    all_dfs, all_summaries = run_parameter_sweep(
        ground_truth_path=str(subset_gt_path),
        db_path=str(db_path),
        masks_dir=str(REPO / 'data/street_view/masks'),
        configs=sweep_configs,
        limit=0,
        chunk_rows=4000,
        spatial_stride=5,
        sample_batch_size=8,
    )

    # Save summaries
    with open(sweep_ckpt, 'w') as f:
        json.dump(all_summaries, f, indent=2, default=str)

    # Save per-config CSVs
    for cfg_name, df in all_dfs.items():
        df.to_csv(STUDY_DIR / f'sweep_{cfg_name}.csv', index=False)

    print('\nSweep complete.')

# Print summary table
print(f'\n{"Config":25s} {"N":>4s} {"Med err":>8s} {"top1@500m":>10s} {"top1@200m":>10s} {"top1@100m":>10s}')
print('-' * 70)
for cfg_name, s in sorted(all_summaries.items()):
    n = s.get('n_samples', 0)
    med = s.get('median_error_m', 0)
    a500 = s.get('top1_acc_500m', 0)
    a200 = s.get('top1_acc_200m', 0)
    a100 = s.get('top1_acc_100m', 0)
    print(f'{cfg_name:25s} {n:4d} {med:8.0f}m {a500:9.1f}% {a200:9.1f}% {a100:9.1f}%')

## Phase 5: Generate Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# CDF of errors
for name, color, label in [
    ('results_sv_baseline.csv', 'steelblue', 'SV (off-grid)'),
    ('results_syn_baseline.csv', 'orange', 'Synthetic (on-grid, upper bound)'),
]:
    p = STUDY_DIR / name
    if p.exists():
        errors = pd.read_csv(p)['error_m'].dropna().sort_values()
        axes[0].plot(errors, np.linspace(0, 1, len(errors)), label=label, color=color)
axes[0].axvline(500, color='red', ls='--', alpha=0.5, label='500m')
axes[0].axvline(1000, color='gray', ls='--', alpha=0.5, label='1000m')
axes[0].set_xlabel('Error (m)')
axes[0].set_ylabel('CDF')
axes[0].set_title('Error Distribution')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Accuracy table
thresholds = [50, 100, 200, 500, 1000]
x_pos = np.arange(len(thresholds))
for name, color, label in [
    ('results_sv_baseline.csv', 'steelblue', 'SV'),
    ('results_syn_baseline.csv', 'orange', 'Synthetic'),
]:
    p = STUDY_DIR / name
    if p.exists():
        errors = pd.read_csv(p)['error_m'].dropna()
        accs = [100 * np.mean(errors <= t) for t in thresholds]
        axes[1].bar(x_pos + (0.15 if 'syn' in name else -0.15), accs, 0.3,
                    label=label, color=color)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'{t}m' for t in thresholds])
axes[1].set_ylabel('Top-1 Accuracy (%)')
axes[1].set_title('Accuracy at Thresholds')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

# Sweep ablation (key configs)
key_configs = ['both_default', 'alt_only', 'compass_only', 'no_sensors']
labels = ['Alt+Compass', 'Alt Only', 'Compass Only', 'No Sensors']
vals = []
for cfg in key_configs:
    if cfg in all_summaries:
        vals.append(all_summaries[cfg].get('top1_acc_500m', 0))
    else:
        vals.append(0)
axes[2].barh(labels[::-1], vals[::-1], color='steelblue')
axes[2].set_xlabel('Top-1 Accuracy @ 500m (%)')
axes[2].set_title('Sensor Ablation')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(STUDY_DIR / 'figures.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures to', STUDY_DIR / 'figures.png')

In [ ]:
print('\n=== FINAL SUMMARY ===')
print(f'Study dir: {STUDY_DIR}')
print(f'\nSV baseline: {sv_result_path}')
print(f'Syn baseline: {syn_result_path}')
print(f'Sweep results: {sweep_ckpt}')

# List all files
for p in sorted(STUDY_DIR.iterdir()):
    if p.is_file():
        print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')